In [1]:
import organizar as orgn, funcoes as fn

laureados = orgn.laureados()
nobelPorNac = orgn.nobelPorNac(laureados)
democracias = orgn.democracias()


/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from datetime import datetime
import wbdata

laureados_2013_2021 = laureados.query("ano >= 2013 and ano <= 2021").copy()

laureados_2013_2021.rename(columns={"pais_nasc" : "pais"}, inplace=True)

laureados_2013_2021["pais"] = laureados_2013_2021["pais"].replace({
    "USA": "United States",
    "the Netherlands": "Netherlands",
    "USSR (now Russia)": "Russia",
    "British Mandate of Palestine (now Israel)": "Israel",
    "Belgian Congo (now Democratic Republic of the Congo)": "Congo (Kinshasa)",
    "Scotland": "United Kingdom"
})

nobelPorAno = (
    laureados_2013_2021
    .groupby(["pais", "ano"])
    .size()
    .reset_index(name="premios")
)

df = democracias.merge(
    nobelPorAno,
    on=["pais", "ano"],
    how="left"
)

df = df.merge(
    dfPop,
    on=["pais", "ano"],
    how="left"
)

df["premios"] = df["premios"].fillna(0)


In [7]:
df.query("pais == 'Brazil'")

,index,pais,ano,pontos,premios,populacao
24,224,Brazil,2013,81,0.0,198478299.0
233,223,Brazil,2014,81,0.0,200085127.0
442,222,Brazil,2015,81,0.0,201675532.0
652,221,Brazil,2016,81,0.0,203218114.0
862,220,Brazil,2017,79,0.0,204703445.0
1071,219,Brazil,2018,78,0.0,206107261.0
1280,218,Brazil,2019,75,0.0,207455459.0
1489,217,Brazil,2020,75,0.0,208660842.0
1699,216,Brazil,2021,74,0.0,209550294.0


In [4]:
import wbdata
from datetime import datetime

indicadores = {
    "SP.POP.TOTL": "populacao"
}

dfPop = wbdata.get_dataframe(
    indicadores,
    date=(datetime(2013,1,1), datetime(2021,12,31))
)

dfPop = dfPop.reset_index()

# Extrai apenas o nome do país (string) caso venha como objeto/lista
dfPop["country"] = dfPop["country"].apply(
    lambda x: x["value"] if isinstance(x, dict)
    else (x[0] if isinstance(x, list) else str(x))
)

dfPop.rename(columns={
    "country": "pais",
    "date": "ano"
}, inplace=True)

dfPop["ano"] = dfPop["ano"].astype(int)

# Remove agregados regionais do Banco Mundial (ex: "World", "Europe & Central Asia")
dfPop = dfPop.dropna(subset=["populacao"])

In [ ]:
laureados_2013_2021 = laureados.query("ano >= 2013 and ano <= 2021")